In [1]:
import numpy as np
import pymc as pm
import arviz as az


/Users/ania/miniforge3/envs/pymc_env/lib/python3.14/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [2]:

def fit_public_model(
    Y_matrix: np.ndarray,      # shape (n_clones, n_individuals)
    depths: np.ndarray,        # shape (n_individuals,)
    min_shared: int = 3,
    draws: int = 2000,
    tune: int = 2000,
    target_accept: float = 0.9,
):
    """
    Bayesian occupancy model correcting for sequencing depth.

    Returns:
        posterior object
        probability each clone is shared in >= min_shared individuals
    """

    n_clones, n_individuals = Y_matrix.shape

    # Depth-based detection probability
    # small pseudo-frequency assumption
    f_bar = 1e-5
    p_detect = 1 - (1 - f_bar) ** depths
    p_detect = np.clip(p_detect, 1e-6, 1 - 1e-6)

    with pm.Model() as model:

        # Hyperprior for sharing probability
        a = pm.Exponential("a", 1.0)
        b = pm.Exponential("b", 1.0)

        # Clone-level true presence probability
        pi = pm.Beta("pi", alpha=a, beta=b, shape=n_clones)

        # Detection probability per individual
        p = pm.Data("p", p_detect)

        # Marginalized likelihood
        prob_obs = pi[:, None] * p[None, :]
        prob_obs = pm.math.clip(prob_obs, 1e-9, 1 - 1e-9)

        Y = pm.Bernoulli(
            "Y",
            p=prob_obs,
            observed=Y_matrix
        )

        trace = pm.sample(
            draws=draws,
            tune=tune,
            target_accept=target_accept,
            chains=4,
            cores=4,
        )

    # ---- Step 5: Compute probability shared >= N ----

    pi_samples = trace.posterior["pi"].values  # shape: chains, draws, clones
    pi_samples = pi_samples.reshape(-1, n_clones)

    prob_shared = []

    for j in range(n_clones):
        # For each posterior sample, simulate sharing count
        shared_counts = np.random.binomial(
            n_individuals,
            pi_samples[:, j]
        )
        prob = np.mean(shared_counts >= min_shared)
        prob_shared.append(prob)

    prob_shared = np.array(prob_shared)

    return trace, prob_shared

# 📊 Calling Public Clones (Step 5)


In [2]:
import pandas as pd

# Load the merged CDR3 data
filepath = "../results/merged/merged_aminoAcid_vFamilyName_jGeneName_productive.tsv.gz"
df = pd.read_csv(filepath, sep="\t", compression="gzip")

print(f"Loaded shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Set index columns that describe CDR3 sequences
index_cols = ["aminoAcid", "vFamilyName", "jGeneName"]
df_indexed = df.set_index(index_cols)

# Extract sample columns (all remaining columns are counts)
Y_counts = df_indexed.values  # shape: (n_clones, n_samples)

# Convert to binary presence/absence matrix (1 if count > 0, else 0)
Y_matrix = (Y_counts > 0).astype(int)

# Get sample names (individual IDs)
sample_names = df_indexed.columns.tolist()
depths = Y_matrix.sum(axis=0)  # number of CDR3s detected per sample

n_clones, n_individuals = Y_matrix.shape
print(f"\nY_matrix shape: {Y_matrix.shape}")
print(f"  n_clones: {n_clones}")
print(f"  n_individuals: {n_individuals}")
print(f"  Sample names: {sample_names}")
print(f"  Depths (CDR3s per sample): {depths}")


Loaded shape: (6672520, 125)
Columns: ['aminoAcid', 'vFamilyName', 'jGeneName', 'F28396E', 'F28396N', 'F20790N', 'F20790E', 'F18072N', 'F27707N', 'F18072E', 'F27707E', 'F15625N', 'F16018E', 'F15625E', 'F16018N', 'F34137N', 'F48016E', 'F42554E', 'F34137E', 'F35497N', 'F35497E', 'F38025E', 'F46506E', 'F38025N', 'F48016N', 'F40405N', 'F40405E', 'F51955N', 'F52014E', 'F50642N', 'F50642E', 'F49984N', 'F5302E', 'F42554N', 'F53955E', 'F49984E', 'F5302N', 'F52014N', 'F46506N', 'F53955N', 'F51955E', 'F69340N', 'F69838N', 'F71547E', 'F71547N', 'F69340E', 'F70393E', 'F6826N', 'F71808N', 'F6826E', 'F52231N', 'F72011N', 'F52231E', 'F72011E', 'F72248N', 'F72520N', 'F72520E', 'F73156N', 'F69838E', 'F73156E', 'F71808E', 'F72248E', 'F72389N', 'F75078E', 'F74622N', 'F70393N', 'F74622E', 'F72389E', 'F76738E', 'F76121E', 'F76738N', 'T02877E', 'F75078N', 'T03194N', 'F76121N', 'T02600N', 'T03445N', 'T02600E', 'T02877N', 'T04634E', 'T03203E', 'T03194E', 'T03203N', 'T03445E', 'T04179E', 'T04348N', 'T04918E', 

Bayesian approach is not feasible in this way - I run it on a subset of samples and after 24h just stopped.

In [7]:
# Get subset of Y_matrix with samples ending in 'E'
mask_E = np.array([name.endswith('E') for name in sample_names])
Y_matrix_E = Y_matrix[:, mask_E]
sample_names_E = [sample_names[i] for i in np.where(mask_E)[0]]
depths_E = depths[mask_E]

print(f"Samples ending with 'E': {sample_names_E}")
print(f"Y_matrix_E shape: {Y_matrix_E.shape}")
print(f"Depths_E: {depths_E}")

# View a small subset
Y_matrix_E[0:3, 0:4]


Samples ending with 'E': ['F28396E', 'F20790E', 'F18072E', 'F27707E', 'F16018E', 'F15625E', 'F48016E', 'F42554E', 'F34137E', 'F35497E', 'F38025E', 'F46506E', 'F40405E', 'F52014E', 'F50642E', 'F5302E', 'F53955E', 'F49984E', 'F51955E', 'F71547E', 'F69340E', 'F70393E', 'F6826E', 'F52231E', 'F72011E', 'F72520E', 'F69838E', 'F73156E', 'F71808E', 'F72248E', 'F75078E', 'F74622E', 'F72389E', 'F76738E', 'F76121E', 'T02877E', 'T02600E', 'T04634E', 'T03203E', 'T03194E', 'T03445E', 'T04179E', 'T04918E', 'T04348E', 'T06003E', 'T08205E', 'T05231E', 'T06306E', 'T13372E', 'T07914E', 'T11566E', 'T12503E', 'T13832E', 'T07325E', 'T13459E', 'T14358E', 'T14976E', 'T14274E', 'T13494E', 'T14758E', 'T08535E']
Y_matrix_E shape: (6672520, 61)
Depths_E: [ 42475  21033  36894  33019  47781  13575  28054  33759  40312  28052
  22659  36513  19473  39464  46236  32832  22089  31538  77197  14788
  30341  33829  83891  46291  51283   8802  29628  41759  90463  44021
  85711  35186  20636  81777  94580  19136  26388 

array([[0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]])

In [12]:
print(np.where(mask_E))
print(np.where(mask_E)[0])

(array([  0,   3,   6,   7,   9,  10,  13,  14,  15,  17,  18,  19,  23,
        25,  27,  29,  31,  32,  37,  40,  42,  43,  46,  49,  50,  53,
        55,  56,  57,  58,  60,  63,  64,  65,  66,  68,  74,  76,  77,
        78,  80,  81,  83,  84,  88,  90,  92,  94,  95,  98,  99, 101,
       103, 105, 108, 109, 112, 115, 117, 118, 120]),)
[  0   3   6   7   9  10  13  14  15  17  18  19  23  25  27  29  31  32
  37  40  42  43  46  49  50  53  55  56  57  58  60  63  64  65  66  68
  74  76  77  78  80  81  83  84  88  90  92  94  95  98  99 101 103 105
 108 109 112 115 117 118 120]


In [ ]:
trace, prob_shared = fit_public_model(Y_matrix_E, depths_E, min_shared=5)
# I am doing it now for a matrix of E. I need to split by cells, then matrix by genotype.
# extract CDR3s which are shared for each genotype, look on their sharing in another genotype.
public_mask = prob_shared >= 0.9
public_clones = np.where(public_mask)[0]

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, pi]


/Users/ania/miniforge3/envs/pymc_env/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Another approach:
Rarefy each repertoire to the same depth (minimum depth across samples) - now I have 25 x for each sample
for N 7...20, compute publicity score = a fraction of rarefactions when present in at least N individuals (in general, in one genotype only).

Repeat K times (e.g., 100–500)

For each clonotype, compute:

Publicity score
=
fraction of rarefactions where present in ≥ N individuals
Publicity score=fraction of rarefactions where present in ≥ N individuals

separate subsampling from the "main" one
- take all E samples
    -subsample each one to depth M
    -for each clonotype, record in how many individuals shared (totla/each of 3 genotypes) clonotype x 4 numbers matrix 
    -repeat K times
